In [1]:
# Cell 1 — Download the SQLite DB (updated weekly, no dhlottery.co.kr needed)
!wget -q -O lotto_data.db \
  "https://github.com/happylie/lotto_data/raw/main/lotto_data.db"
print("Downloaded!")

Downloaded!


In [2]:
# Cell 2 — Export to CSV
import sqlite3, pandas as pd

con = sqlite3.connect("lotto_data.db")
df = pd.read_sql("SELECT * FROM tb_lotto_list ORDER BY round ASC", con)
con.close()

# Rename columns to English
df.columns = ["draw_no", "draw_date", "b1", "b2", "b3", "b4", "b5", "b6", "bonus"]

print(f"Draws: {len(df)}  (1회 ~ {df['draw_no'].max()}회)")
print(df.head(3))
print("...")
print(df.tail(3))

df.to_csv("lotto_data.csv", index=False, encoding="utf-8-sig")
print("\nSaved → lotto_data.csv")

Draws: 1204  (1회 ~ 1204회)
   draw_no   draw_date  b1  b2  b3  b4  b5  b6  bonus
0        1   2002.12.7  10  23  29  33  37  40     16
1        2  2002.12.14   9  13  21  25  32  42      2
2        3  2002.12.21  11  16  19  21  27  31     30
...
      draw_no    draw_date  b1  b2  b3  b4  b5  b6  bonus
1201     1202  2025.12.13    5  12  21  33  37  40      7
1202     1203  2025.12.20    3   6  18  29  35  39     24
1203     1204  2025.12.27    8  16  28  30  31  44     27

Saved → lotto_data.csv


In [3]:
pip install pandas numpy scipy matplotlib koreanize-matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 33.7 MB/s eta 0:00:00


In [4]:
%%writefile lotto_forensics_new.py
"""
lotto_forensics_new.py
==================
Statistical fairness audit for Korean Lotto 6/45.

Implements eleven tests — some ported/reformulated from election-forensics
methodology, some natural to lottery auditing.

  L1  Ball-frequency chi-square (χ²(44))            ← most fundamental
  L2  Last-digit test — corrected non-uniform null     ← ported F2
  L3  Draw-sum distribution test (exact theoretical)   ← natural
  L4  Odd/even & high/low split tests (hypergeometric) ← natural
  L5  Pairwise co-occurrence test (990 pairs)          ← natural
  L6  0s-and-5s mechanical-bias test                   ← ported F8
  L7  Last-digit mean & variance (corrected null)      ← ported F9
  L8  Simulation-adjusted chi-square (S&M 2011)        ← ported F7d
  L9  Temporal autocorrelation (per ball, Ljung-Box)   ← natural
  L10 Summary-statistic variance test                  ← ported F3
  L11 Cross-ball correlation matrix (Fisher Z test)    ← natural

Usage
-----
  python lotto_forensics_new.py lotto_data.csv                  # all draws
  python lotto_forensics_new.py lotto_data.csv --from 500       # draw 500 onwards
  python lotto_forensics_new.py lotto_data.csv --to 999         # up to draw 999
  python lotto_forensics_new.py lotto_data.csv --from 500 --to 999  # range
  python lotto_forensics_new.py lotto_data.csv --lag 10         # custom lag period
  python lotto_forensics_new.py                                 # synthetic demo

CSV format (auto-detected column names):
  draw_no, date, b1, b2, b3, b4, b5, b6[, bonus]
  — or ball1..ball6 / num1..num6 — encoding utf-8 or cp949

Dependencies
------------
  pip install pandas numpy scipy matplotlib koreanize-matplotlib
"""

import argparse
import math
import sys
import warnings
from itertools import combinations

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import chisquare, hypergeom, norm
from scipy.stats import chi2 as chi2_dist

warnings.filterwarnings("ignore")

try:
    import koreanize_matplotlib  # noqa: F401
except ImportError:
    pass

matplotlib.rcParams["axes.unicode_minus"] = False

# ─────────────────────────────────────────────
# GLOBAL CONSTANTS
# ─────────────────────────────────────────────
N_POOL = 45           # number pool
K_DRAW = 6            # balls per draw
C_N_K  = math.comb(N_POOL, K_DRAW)   # 8,145,060

_LD_POP = {d: 0 for d in range(10)}
for _b in range(1, N_POOL + 1):
    _LD_POP[_b % 10] += 1

EXPECTED_LD_PROPS = np.array([_LD_POP[d] / N_POOL for d in range(10)])
EXPECTED_LD_MEAN  = sum(d * _LD_POP[d] for d in range(10)) / N_POOL
EXPECTED_LD_VAR   = (
    sum(d**2 * _LD_POP[d] for d in range(10)) / N_POOL - EXPECTED_LD_MEAN**2
)

EXPECTED_SUM_MEAN = K_DRAW * (N_POOL + 1) / 2
POP_VAR           = (N_POOL**2 - 1) / 12
EXPECTED_SUM_VAR  = (
    K_DRAW * POP_VAR * (N_POOL - K_DRAW) / (N_POOL - 1)
)

N_ODD  = 23
N_EVEN = 22
N_HIGH = 23
N_LOW  = 22

P_COOCCUR   = math.comb(N_POOL - 2, K_DRAW - 2) / C_N_K
P_ZERO_FIVE = 9 / N_POOL


# ─────────────────────────────────────────────
# DATA LOADING & FILTERING
# ─────────────────────────────────────────────
def load_lotto_data(path: str):
    """
    Read lotto draw history from CSV.
    Returns (DataFrame, ball_col_list[6], bonus_col_or_None).
    """
    try:
        df = pd.read_csv(path, encoding="utf-8")
    except UnicodeDecodeError:
        df = pd.read_csv(path, encoding="cp949")

    df.columns = (
        df.columns.str.strip().str.lower()
        .str.replace(" ", "_").str.replace("번호", "")
    )

    for pattern in [
        lambda c: len(c) == 2 and c[0] == "b" and c[1].isdigit(),
        lambda c: c.startswith("ball") and c[4:].isdigit(),
        lambda c: c.startswith("num")  and c[3:].isdigit(),
        lambda c: len(c) == 2 and c[0] == "n" and c[1].isdigit(),
        lambda c: c.startswith("당첨"),
    ]:
        cols = sorted([c for c in df.columns if pattern(c)])
        if len(cols) >= 6:
            ball_cols = cols[:6]
            break
    else:
        raise ValueError(
            f"Cannot identify ball columns in: {list(df.columns)}\n"
            "Rename them to b1..b6 or ball1..ball6."
        )

    bonus_candidates = [c for c in df.columns if "bonus" in c or "보너스" in c]
    bonus_col = bonus_candidates[0] if bonus_candidates else None

    for c in ball_cols + ([bonus_col] if bonus_col else []):
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df.dropna(subset=ball_cols).reset_index(drop=True)

    for draw_col in ["draw_no", "회차", "round", "draw"]:
        if draw_col in df.columns:
            df = df.sort_values(draw_col).reset_index(drop=True)
            break

    print(f"  Loaded {len(df):,} draws | balls: {ball_cols} | bonus: {bonus_col}")
    return df, ball_cols, bonus_col


def filter_by_draw_range(
    df: pd.DataFrame,
    draw_from: int | None,
    draw_to:   int | None,
) -> pd.DataFrame:
    """
    Keep only rows whose draw_no is within [draw_from, draw_to].
    """
    draw_col = None
    for candidate in ["draw_no", "회차", "round", "draw"]:
        if candidate in df.columns:
            draw_col = candidate
            break

    if draw_col is None:
        if draw_from is not None or draw_to is not None:
            raise ValueError(
                "Cannot filter by draw range — no draw_no column found in CSV."
            )
        return df

    if draw_from is not None:
        df = df[df[draw_col] >= draw_from]
    if draw_to is not None:
        df = df[df[draw_col] <= draw_to]

    df = df.reset_index(drop=True)

    if len(df) == 0:
        lo = draw_from if draw_from is not None else "start"
        hi = draw_to   if draw_to   is not None else "end"
        raise ValueError(f"No draws found in range [{lo}, {hi}].")

    lo_actual = int(df[draw_col].min())
    hi_actual = int(df[draw_col].max())
    print(f"  Filtered to draws {lo_actual}–{hi_actual}  ({len(df):,} draws retained)")
    return df


def extract_balls(df: pd.DataFrame, ball_cols: list) -> np.ndarray:
    return df[ball_cols].values.astype(int)


def generate_synthetic_data(n_draws: int = 1100, seed: int = 42):
    rng  = np.random.default_rng(seed)
    pool = np.arange(1, N_POOL + 1)
    rows = []
    for i in range(n_draws):
        draw  = sorted(rng.choice(pool, K_DRAW, replace=False))
        bonus = int(rng.choice([b for b in pool if b not in draw]))
        rows.append([i + 1, f"draw_{i+1}"] + draw + [bonus])
    df = pd.DataFrame(
        rows,
        columns=["draw_no", "date", "b1", "b2", "b3", "b4", "b5", "b6", "bonus"],
    )
    return df, ["b1", "b2", "b3", "b4", "b5", "b6"], "bonus"


# ─────────────────────────────────────────────
# THEORETICAL TOOLS
# ─────────────────────────────────────────────
def compute_exact_sum_distribution() -> dict:
    max_s = sum(range(N_POOL - K_DRAW + 1, N_POOL + 1))
    dp    = np.zeros((K_DRAW + 1, max_s + 1), dtype=np.int64)
    dp[0, 0] = 1
    for ball in range(1, N_POOL + 1):
        for j in range(min(ball, K_DRAW), 0, -1):
            end = max_s + 1 - ball
            if end > 0:
                dp[j, ball : max_s + 1] += dp[j - 1, :end]
    return {
        s: int(dp[K_DRAW, s]) / C_N_K
        for s in range(max_s + 1)
        if dp[K_DRAW, s] > 0
    }


def merge_bins(obs_arr, exp_arr, min_exp=5.0):
    merged_obs, merged_exp = [], []
    acc_o, acc_e = 0.0, 0.0
    for o, e in zip(obs_arr, exp_arr):
        acc_o += o
        acc_e += e
        if acc_e >= min_exp:
            merged_obs.append(acc_o)
            merged_exp.append(acc_e)
            acc_o, acc_e = 0.0, 0.0

    if acc_o > 0 or acc_e > 0:
        if merged_obs:
            merged_obs[-1] += acc_o
            merged_exp[-1] += acc_e
        else:
            merged_obs.append(acc_o)
            merged_exp.append(acc_e)

    total_obs = sum(merged_obs)
    total_exp = sum(merged_exp)
    if total_exp > 0:
        merged_exp = [e * (total_obs / total_exp) for e in merged_exp]

    return merged_obs, merged_exp


# ─────────────────────────────────────────────
# TEST IMPLEMENTATIONS
# ─────────────────────────────────────────────

def test_L1_ball_frequency(balls: np.ndarray, logs: list) -> dict:
    log = lambda m: (print(m), logs.append(m))
    log("\n" + "=" * 62)
    log("  [L1] Ball-Frequency Chi-Square  χ²(44)")
    log("=" * 62)

    n_draws = len(balls)
    exp     = n_draws * K_DRAW / N_POOL

    counts = np.zeros(N_POOL, dtype=int)
    for row in balls:
        counts[row - 1] += 1

    exp_arr = np.full(N_POOL, exp)
    exp_arr = exp_arr * (np.sum(counts) / np.sum(exp_arr))

    chi2_stat, p_val = chisquare(counts, f_exp=exp_arr)

    log(f"  N draws              : {n_draws:,}")
    log(f"  Expected per ball    : {exp:.2f}")
    log(f"  χ²(44) = {chi2_stat:.4f}   p = {p_val:.4f}")
    log("  Result: " + ("PASS" if p_val > 0.05 else "FAIL ★"))

    p_appear = K_DRAW / N_POOL
    se       = math.sqrt(n_draws * p_appear * (1 - p_appear))
    zscores  = (counts - exp) / se
    top3     = np.argsort(np.abs(zscores))[::-1][:3]
    log("  Top-3 outlier balls (|z|-ranked):")
    for idx in top3:
        log(f"    Ball {idx+1:2d}: count={counts[idx]:4d}  z={zscores[idx]:+.3f}")

    return {"counts": counts, "expected": exp, "chi2": chi2_stat, "p": p_val,
            "zscores": zscores}


def test_L2_last_digit(balls: np.ndarray, logs: list) -> dict:
    log = lambda m: (print(m), logs.append(m))
    log("\n" + "=" * 62)
    log("  [L2] Last-Digit Test — Corrected Null for 1–45")
    log("=" * 62)

    flat = balls.flatten()
    ld   = flat % 10
    n    = len(ld)
    obs  = np.array([np.sum(ld == d) for d in range(10)])
    exp  = EXPECTED_LD_PROPS * n

    exp = exp * (np.sum(obs) / np.sum(exp))
    chi2_stat, p_val = chisquare(obs, f_exp=exp)

    log(f"  N ball appearances   : {n:,}")
    log(f"  χ²(9) = {chi2_stat:.4f}   p = {p_val:.4f}")
    log("  Result: " + ("PASS" if p_val > 0.05 else "FAIL ★"))
    log("  digit  observed  expected  ratio")
    for d in range(10):
        log(f"    {d}     {obs[d]:5d}   {exp[d]:7.1f}  {obs[d]/exp[d]:.3f}")

    return {"obs": obs, "exp": exp, "chi2": chi2_stat, "p": p_val}


def test_L3_draw_sum(balls: np.ndarray, logs: list) -> dict:
    log = lambda m: (print(m), logs.append(m))
    log("\n" + "=" * 62)
    log("  [L3] Draw-Sum Distribution Test")
    log("=" * 62)

    sums    = balls.sum(axis=1)
    n_draws = len(sums)

    obs_mean = float(sums.mean())
    obs_std  = float(sums.std())

    se_mean  = (EXPECTED_SUM_VAR / n_draws) ** 0.5
    z_mean   = (obs_mean - EXPECTED_SUM_MEAN) / se_mean
    p_mean   = 2 * (1 - norm.cdf(abs(z_mean)))

    log(f"  N draws              : {n_draws:,}")
    log(f"  Observed mean        : {obs_mean:.3f}   (expected {EXPECTED_SUM_MEAN:.1f})")
    log(f"  (a) Z-test on mean   : z = {z_mean:+.4f}   p = {p_mean:.4f}")

    theo = compute_exact_sum_distribution()
    all_s = sorted(theo)
    obs_cnt = np.array([np.sum(sums == s) for s in all_s])
    exp_cnt = np.array([theo[s] * n_draws  for s in all_s])

    m_obs, m_exp = merge_bins(obs_cnt, exp_cnt, min_exp=5.0)
    chi2_stat, p_chi2 = chisquare(m_obs, f_exp=m_exp)
    df_chi = len(m_obs) - 1

    log(f"  (b) χ²({df_chi}) binned-sum test = {chi2_stat:.4f}   p = {p_chi2:.4f}  "
        + ("PASS" if p_chi2 > 0.05 else "FAIL ★"))

    return {
        "sums": sums, "theo_dist": theo,
        "obs_mean": obs_mean, "obs_std": obs_std,
        "z_mean": z_mean, "p_mean": p_mean,
        "chi2": chi2_stat, "p": p_chi2,
    }


def test_L4_splits(balls: np.ndarray, logs: list) -> dict:
    log = lambda m: (print(m), logs.append(m))
    log("\n" + "=" * 62)
    log("  [L4] Odd/Even & High/Low Split Tests  (Hypergeometric Null)")
    log("=" * 62)

    n_draws = len(balls)
    out = {}

    configs = [
        ("Odd  (1,3,…,45)",   lambda r: r % 2 == 1,  N_ODD),
        ("High (23–45)",       lambda r: r >= 23,     N_HIGH),
    ]
    for label, cond, n_pop in configs:
        per_draw = np.array([np.sum(cond(row)) for row in balls])
        obs      = np.bincount(per_draw, minlength=K_DRAW + 1)[: K_DRAW + 1]
        hg       = hypergeom(N_POOL, n_pop, K_DRAW)
        exp      = np.array([hg.pmf(k) * n_draws for k in range(K_DRAW + 1)])
        m_obs, m_exp = merge_bins(obs, exp)
        chi2_stat, p_val = chisquare(m_obs, f_exp=m_exp)
        df_val = len(m_obs) - 1

        log(f"\n  [{label}]  (N_success_in_pool = {n_pop})")
        log(f"  χ²({df_val}) = {chi2_stat:.4f}   p = {p_val:.4f}  "
            + ("PASS" if p_val > 0.05 else "FAIL ★"))

        out[label] = {"per_draw": per_draw, "obs": obs, "exp": exp,
                      "chi2": chi2_stat, "p": p_val}

    return out


def test_L5_pairwise(balls: np.ndarray, logs: list) -> dict:
    log = lambda m: (print(m), logs.append(m))
    log("\n" + "=" * 62)
    log("  [L5] Pairwise Co-Occurrence Test  (990 pairs)")
    log("=" * 62)

    n_draws = len(balls)
    exp_per_pair = n_draws * P_COOCCUR

    pair_idx = {p: i for i, p in enumerate(combinations(range(1, N_POOL + 1), 2))}
    counts   = np.zeros(len(pair_idx), dtype=int)

    for row in balls:
        for pair in combinations(sorted(row), 2):
            counts[pair_idx[pair]] += 1

    exp_arr   = np.full(len(counts), exp_per_pair)
    exp_arr = exp_arr * (np.sum(counts) / np.sum(exp_arr))

    chi2_stat, p_val = chisquare(counts, f_exp=exp_arr)
    df_val = len(counts) - 1

    se      = math.sqrt(exp_per_pair * (1 - P_COOCCUR))
    zscores = (counts - exp_per_pair) / se

    log(f"  Expected per pair: {exp_per_pair:.2f}  (se ≈ {se:.2f})")
    log(f"  χ²({df_val}) = {chi2_stat:.4f}   p = {p_val:.4f}  "
        + ("PASS" if p_val > 0.05 else "FAIL ★"))

    return {
        "counts": counts, "pairs": list(pair_idx.keys()), "expected": exp_per_pair,
        "zscores": zscores, "chi2": chi2_stat, "p": p_val,
    }


def test_L6_zeros_fives(balls: np.ndarray, logs: list) -> dict:
    log = lambda m: (print(m), logs.append(m))
    log("\n" + "=" * 62)
    log("  [L6] 0s-and-5s Test  (Mechanical Bias Indicator, ported F8)")
    log("=" * 62)

    flat  = balls.flatten()
    n     = len(flat)
    count = int(np.sum((flat % 10 == 0) | (flat % 10 == 5)))
    p_hat = count / n
    se    = math.sqrt(P_ZERO_FIVE * (1 - P_ZERO_FIVE) / n)
    z     = (p_hat - P_ZERO_FIVE) / se
    p_val = 2 * (1 - norm.cdf(abs(z)))

    log(f"  Z-statistic          : {z:+.4f}")
    log(f"  p-value              : {p_val:.4f}  " + ("PASS" if p_val > 0.05 else "FAIL ★"))

    return {"count_05": count, "p_hat": p_hat, "z": z, "p": p_val}


def test_L7_ld_mean_var(balls: np.ndarray, logs: list) -> dict:
    log = lambda m: (print(m), logs.append(m))
    log("\n" + "=" * 62)
    log("  [L7] Last-Digit Mean & Variance  (Corrected Null, ported F9)")
    log("=" * 62)

    flat = balls.flatten()
    ld   = flat % 10
    n    = len(ld)

    obs_mean = float(ld.mean())
    obs_var  = float(ld.var())

    se_mean  = math.sqrt(EXPECTED_LD_VAR / n)
    z_mean   = (obs_mean - EXPECTED_LD_MEAN) / se_mean
    p_mean   = 2 * (1 - norm.cdf(abs(z_mean)))

    chi2_var  = (n - 1) * obs_var / EXPECTED_LD_VAR
    p_var_low = chi2_dist.cdf(chi2_var, df=n - 1)
    p_var     = 2 * min(p_var_low, 1 - p_var_low)

    log(f"  Z (mean)             : {z_mean:+.4f}   p = {p_mean:.4f}  "
        + ("PASS" if p_mean > 0.05 else "FAIL ★"))
    log(f"  χ² (variance)        : {chi2_var:.2f}   p = {p_var:.4f}  "
        + ("PASS" if p_var > 0.05 else "FAIL ★"))

    return {
        "obs_mean": obs_mean, "obs_var": obs_var,
        "z_mean": z_mean, "p_mean": p_mean,
        "chi2_var": chi2_var, "p_var": p_var,
    }


def test_L8_sim_chi2(balls: np.ndarray, logs: list, n_sim: int = 500, batch: int = 100) -> dict:
    log = lambda m: (print(m), logs.append(m))
    log("\n" + "=" * 62)
    log(f"  [L8] Simulation-Adjusted Chi-Square  (S&M 2011, ported F7d)")
    log("=" * 62)

    n_draws    = len(balls)
    exp        = n_draws * K_DRAW / N_POOL
    counts_obs = np.zeros(N_POOL, dtype=int)
    for row in balls:
        counts_obs[row - 1] += 1
    obs_chi2 = float(np.sum((counts_obs - exp) ** 2 / exp))

    rng      = np.random.default_rng(42)
    sim_vals = []
    processed = 0
    while processed < n_sim:
        b        = min(batch, n_sim - processed)
        n_total  = b * n_draws
        rand_mat = rng.random((n_total, N_POOL))
        perm_mat = np.argsort(rand_mat, axis=1)[:, :K_DRAW]
        perm_3d  = perm_mat.reshape(b, n_draws, K_DRAW)
        for i in range(b):
            flat = perm_3d[i].flatten()
            cnts = np.bincount(flat, minlength=N_POOL)
            sim_vals.append(float(np.sum((cnts - exp) ** 2 / exp)))
        processed += b

    sim_arr     = np.array(sim_vals)
    sim_95      = float(np.percentile(sim_arr, 95))
    textbook_95 = chi2_dist.ppf(0.95, df=N_POOL - 1)
    chi2_trans  = obs_chi2 * textbook_95 / sim_95 if sim_95 > 0 else float("nan")
    p_empirical = float(np.mean(sim_arr >= obs_chi2))

    log(f"  χ²_trans             : {chi2_trans:.4f}")
    log(f"  Empirical p-value    : {p_empirical:.4f}  " + ("PASS" if chi2_trans <= textbook_95 else "FAIL ★"))

    return {
        "obs_chi2": obs_chi2, "sim_arr": sim_arr,
        "sim_95": sim_95, "chi2_trans": chi2_trans,
        "textbook_95": textbook_95, "p_empirical": p_empirical,
    }


def test_L9_autocorrelation(balls: np.ndarray, logs: list, max_lag: int = 5) -> dict:
    log = lambda m: (print(m), logs.append(m))
    log("\n" + "=" * 62)
    log(f"  [L9] Temporal Autocorrelation  (lags 1–{max_lag}, per ball)")
    log("=" * 62)

    n_draws = len(balls)
    indicator = np.zeros((n_draws, N_POOL), dtype=np.float64)
    for i, row in enumerate(balls):
        indicator[i, row - 1] = 1.0

    ac_matrix = np.zeros((N_POOL, max_lag))
    for b in range(N_POOL):
        x  = indicator[:, b]
        xc = x - x.mean()
        denom = float(np.dot(xc, xc))
        if denom == 0: continue
        for lag in range(1, max_lag + 1):
            if n_draws - lag > 0:
                r = float(np.dot(xc[lag:], xc[:-lag])) / denom
                ac_matrix[b, lag - 1] = r
            else:
                ac_matrix[b, lag - 1] = 0.0

    lag_vec = np.arange(1, max_lag + 1)
    # Avoid division by zero if lag is somehow larger than draws
    weights = np.where(n_draws - lag_vec > 0, 1.0 / (n_draws - lag_vec), 0)
    lb_stat = n_draws * (n_draws + 2) * float(np.sum(ac_matrix ** 2 * weights[np.newaxis, :]))
    lb_df   = N_POOL * max_lag
    lb_p    = float(1 - chi2_dist.cdf(lb_stat, df=lb_df))

    log(f"  Ljung-Box Q({lb_df}) = {lb_stat:.4f}   p = {lb_p:.4f}  " + ("PASS" if lb_p > 0.05 else "FAIL ★"))

    return {"ac_matrix": ac_matrix, "n_draws": n_draws, "lb_stat": lb_stat, "lb_p": lb_p, "max_lag": max_lag}


def test_L10_variance(balls: np.ndarray, logs: list) -> dict:
    log = lambda m: (print(m), logs.append(m))
    log("\n" + "=" * 62)
    log("  [L10] Summary-Statistic Variance Test  (ported F3)")
    log("=" * 62)

    n_draws = len(balls)
    out     = {}

    def chi2_var_test(label, obs_var, theo_var):
        chi2_v    = (n_draws - 1) * obs_var / theo_var
        p_low     = chi2_dist.cdf(chi2_v, df=n_draws - 1)
        p_twotail = 2 * min(p_low, 1 - p_low)
        log(f"  [{label}] χ²({n_draws-1}) = {chi2_v:.2f}   p = {p_twotail:.4f}  " + ("PASS" if p_twotail > 0.05 else "FAIL ★"))
        return {"obs_var": obs_var, "theo_var": theo_var, "chi2": chi2_v, "p": p_twotail}

    sums = balls.sum(axis=1)
    out["sum"] = chi2_var_test("Sum of 6 balls", float(sums.var(ddof=1)), EXPECTED_SUM_VAR)

    odd_per_draw = np.array([int(np.sum(row % 2 == 1)) for row in balls])
    theo_var_odd = K_DRAW * (N_ODD / N_POOL) * (N_EVEN / N_POOL) * (N_POOL - K_DRAW) / (N_POOL - 1)
    out["odd"] = chi2_var_test("Odd-ball count per draw", float(odd_per_draw.var(ddof=1)), theo_var_odd)

    ranges         = balls.max(axis=1) - balls.min(axis=1)
    rng_mc         = np.random.default_rng(0)
    pool           = np.arange(1, N_POOL + 1)
    mc_ranges      = np.array([np.ptp(rng_mc.choice(pool, K_DRAW, replace=False)) for _ in range(10_000)])
    out["range"]   = chi2_var_test("Draw range (max−min)", float(ranges.var(ddof=1)), float(mc_ranges.var()))

    return out


def test_L11_cross_correlation(balls: np.ndarray, logs: list) -> dict:
    log = lambda m: (print(m), logs.append(m))
    log("\n" + "=" * 62)
    log("  [L11] Cross-Ball Correlation Matrix (Pearson)")
    theo_corr = -1.0 / (N_POOL - 1)
    log(f"  Theoretical structural pair correlation: {theo_corr:.5f} (-1/44)")
    log("=" * 62)

    n_draws = len(balls)
    indicator = np.zeros((n_draws, N_POOL), dtype=np.float64)
    for i, row in enumerate(balls):
        indicator[i, row - 1] = 1.0

    # Calculate Pearson correlation matrix
    corr_matrix = np.corrcoef(indicator.T)

    # Extract the 990 unique upper-triangle correlations
    triu_idx = np.triu_indices(N_POOL, k=1)
    obs_corrs = corr_matrix[triu_idx]

    # Fisher Z-transformation to test significance
    def fisher_z(r):
        r = np.clip(r, -0.9999, 0.9999)
        return 0.5 * np.log((1 + r) / (1 - r))

    z_obs  = fisher_z(obs_corrs)
    z_theo = fisher_z(theo_corr)
    se_z   = 1.0 / math.sqrt(max(n_draws - 3, 1))

    z_stats = (z_obs - z_theo) / se_z
    p_vals  = 2 * (1 - norm.cdf(np.abs(z_stats)))

    # Global heuristic chi-square test (assumes weak dependence)
    chi2_approx = float(np.sum(z_stats**2))
    p_chi2 = float(1 - chi2_dist.cdf(chi2_approx, df=len(obs_corrs)))

    log(f"  Total pairs          : {len(obs_corrs)}")
    log(f"  Approx global χ²({len(obs_corrs)}) = {chi2_approx:.2f}   p = {p_chi2:.4f}")
    log("  Result (Global): " + ("PASS" if p_chi2 > 0.05 else "FAIL ★"))

    return {
        "corr_matrix": corr_matrix,
        "obs_corrs": obs_corrs,
        "z_stats": z_stats,
        "chi2_approx": chi2_approx,
        "p_chi2": p_chi2,
    }


# ─────────────────────────────────────────────
# DASHBOARD
# ─────────────────────────────────────────────
def _pass_fail(p: float, threshold: float = 0.05) -> str:
    return "PASS" if p > threshold else "FAIL ★"

def plot_dashboard(balls: np.ndarray, results: dict, out_path: str = "lotto_forensics_dashboard.png", range_label: str = "") -> None:
    fig, axes = plt.subplots(4, 4, figsize=(22, 20))
    title = "Korean Lotto 6/45 — Statistical Fairness Audit Dashboard"
    if range_label: title += f"  [{range_label}]"
    fig.suptitle(title, fontsize=16, fontweight="bold", y=0.995)

    r1, r2, r3, r4 = results["L1"], results["L2"], results["L3"], results["L4"]
    r5, r6, r7, r8 = results["L5"], results["L6"], results["L7"], results["L8"]
    r9, r10, r11   = results["L9"], results["L10"], results["L11"]
    n_draws = len(balls)
    ball_nums = np.arange(1, N_POOL + 1)
    W = 0.4

    # --- ROW 0 ---
    ax = axes[0, 0]
    clr = ["tomato" if abs(z) > 2 else "steelblue" for z in r1["zscores"]]
    ax.bar(ball_nums, r1["counts"], color=clr, edgecolor="none", alpha=0.85)
    ax.axhline(r1["expected"], color="black", ls="--", lw=1.5, label=f"Exp ({r1['expected']:.0f})")
    ax.set_title(f"[L1] Ball Frequency  χ²(44)={r1['chi2']:.2f}  p={r1['p']:.3f}  {_pass_fail(r1['p'])}", fontsize=9)
    ax.set_xlabel("Ball number", fontsize=8); ax.set_ylabel("Count", fontsize=8)

    ax = axes[0, 1]
    ax.hist(r3["sums"], bins=range(int(r3["sums"].min()), int(r3["sums"].max()) + 2), alpha=0.60, color="steelblue", density=True, label="Observed")
    sv = sorted(r3["theo_dist"])
    ax.plot(sv, [r3["theo_dist"][s] for s in sv], "r-", lw=1.5, label="Exact Theo")
    ax.set_title(f"[L3] Draw Sum  p={r3['p']:.3f}  {_pass_fail(r3['p'])}", fontsize=9)
    ax.set_xlabel("Sum", fontsize=8); ax.legend(fontsize=7)

    ax = axes[0, 2]
    r4k = list(r4.keys())[0]
    k_vals = np.arange(K_DRAW + 1)
    ax.bar(k_vals - W/2, r4[r4k]["obs"], W, alpha=0.85, color="mediumpurple", label="Obs")
    ax.bar(k_vals + W/2, r4[r4k]["exp"], W, alpha=0.65, color="slategray", label="Exp")
    ax.set_title(f"[L4] Odd Count  p={r4[r4k]['p']:.3f}  {_pass_fail(r4[r4k]['p'])}", fontsize=9)
    ax.set_xlabel("# odd balls", fontsize=8); ax.set_xticks(k_vals)

    ax = axes[0, 3]
    r4k2 = list(r4.keys())[1] if len(r4) > 1 else list(r4.keys())[0]
    ax.bar(k_vals - W/2, r4[r4k2]["obs"], W, alpha=0.85, color="teal", label="Obs")
    ax.bar(k_vals + W/2, r4[r4k2]["exp"], W, alpha=0.65, color="slategray", label="Exp")
    ax.set_title(f"[L4] High Count  p={r4[r4k2]['p']:.3f}  {_pass_fail(r4[r4k2]['p'])}", fontsize=9)
    ax.set_xlabel("# high balls", fontsize=8); ax.set_xticks(k_vals)

    # --- ROW 1 ---
    ax = axes[1, 0]
    digits = np.arange(10)
    ax.bar(digits - W/2, r2["obs"], W, alpha=0.85, color="salmon")
    ax.bar(digits + W/2, r2["exp"], W, alpha=0.65, color="slategray")
    ax.set_title(f"[L2] Last-Digit  p={r2['p']:.3f}  {_pass_fail(r2['p'])}", fontsize=9)
    ax.set_xticks(digits)

    ax = axes[1, 1]
    ax.hist(r8["sim_arr"], bins=35, alpha=0.75, color="mediumseagreen", edgecolor="none")
    ax.axvline(r8["obs_chi2"], color="red", lw=2.0, ls="--", label=f"Obs: {r8['obs_chi2']:.2f}")
    ax.set_title(f"[L8] Sim-Adj χ²  emp_p={r8['p_empirical']:.3f}  {_pass_fail(r8['p_empirical'])}", fontsize=9)
    ax.legend(fontsize=7)

    ax = axes[1, 2]
    ax.hist(r5["zscores"], bins=40, alpha=0.75, color="darkcyan", density=True)
    xn = np.linspace(-4, 4, 200)
    ax.plot(xn, norm.pdf(xn), "r-", lw=1.5, label="N(0,1)")
    ax.set_title(f"[L5] Pair Co-occurrence Z  p={r5['p']:.3f}  {_pass_fail(r5['p'])}", fontsize=9)

    ax = axes[1, 3]
    ax.bar(ball_nums, r1["zscores"], color=clr, edgecolor="none", alpha=0.85)
    ax.axhline(0, color="black", lw=0.8)
    ax.axhline(2, color="red", lw=1.0, ls=":", label="±2σ")
    ax.axhline(-2, color="red", lw=1.0, ls=":")
    ax.set_title("[L1] Ball Frequency Z-scores", fontsize=9)

    # --- ROW 2 ---
    ax = axes[2, 0]
    im = ax.imshow(r9["ac_matrix"].T, aspect="auto", cmap="RdBu_r", vmin=-0.15, vmax=0.15, interpolation="nearest")
    fig.colorbar(im, ax=ax, label="Autocorrelation")
    ax.set_title(f"[L9] Temporal Autocorr  p={r9['lb_p']:.3f}  {_pass_fail(r9['lb_p'])}", fontsize=9)

    # Use dynamic ticks up to max_lag, ensuring we don't crowd it if max_lag is massive
    max_lag_display = min(r9["max_lag"], 20)
    ax.set_yticks(range(max_lag_display))
    ax.set_yticklabels([f"Lag {i+1}" for i in range(max_lag_display)], fontsize=7)

    ax = axes[2, 1]
    ac1 = r9["ac_matrix"][:, 0]
    thr = 1.96 / math.sqrt(r9["n_draws"])
    clr3 = ["tomato" if abs(r) > thr else "steelblue" for r in ac1]
    ax.bar(ball_nums, ac1, color=clr3, edgecolor="none", alpha=0.85)
    ax.axhline(thr, color="red", lw=1.0, ls=":")
    ax.axhline(-thr, color="red", lw=1.0, ls=":")
    ax.set_title("[L9] Lag-1 Autocorrelation per Ball", fontsize=9)

    ax = axes[2, 2]
    labels_v = ["Sum", "Odd count", "Range"]
    ratios = [r10["sum"]["obs_var"]/r10["sum"]["theo_var"], r10["odd"]["obs_var"]/r10["odd"]["theo_var"], r10["range"]["obs_var"]/r10["range"]["theo_var"]]
    pvals_v = [r10["sum"]["p"], r10["odd"]["p"], r10["range"]["p"]]
    clr4 = ["tomato" if p < 0.05 else "steelblue" for p in pvals_v]
    bars = ax.bar(labels_v, ratios, color=clr4, edgecolor="none", alpha=0.85)
    ax.axhline(1.0, color="black", lw=1.5, ls="--")
    for bar, p in zip(bars, pvals_v): ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f"p={p:.3f}", ha="center", va="bottom", fontsize=8)
    ax.set_title("[L10] Variance Ratio (obs / theo)", fontsize=9)

    ax = axes[2, 3]
    plot_corr = r11["corr_matrix"].copy()
    np.fill_diagonal(plot_corr, np.nan)
    im2 = ax.imshow(plot_corr, cmap="RdBu_r", vmin=-0.1, vmax=0.1, interpolation="none")
    fig.colorbar(im2, ax=ax, label="Pearson r")
    ax.set_title(f"[L11] Cross-Ball Correlation Heatmap", fontsize=9)
    ax.set_xlabel("Ball number")

    # --- ROW 3 ---
    ax = axes[3, 0]
    ax.hist(r11["z_stats"], bins=40, alpha=0.75, color="purple", density=True, edgecolor="none")
    ax.plot(xn, norm.pdf(xn), "r-", lw=1.5, label="N(0,1)")
    ax.set_title(f"[L11] Cross-Ball Z-scores  p={r11['p_chi2']:.3f}  {_pass_fail(r11['p_chi2'])}", fontsize=9)
    ax.set_xlabel("Fisher Z-score")

    axes[3, 1].axis("off")
    axes[3, 2].axis("off")

    ax = axes[3, 3]
    ax.axis("off")
    def _pline(tag, label, p): return f"  {'✓' if p > 0.05 else '✗'} {tag:<4} {label:<28} p={p:.4f}"
    lines = [
        "══════ AUDIT SUMMARY ══════",
        _pline("L1",  "Ball frequency",          r1["p"]),
        _pline("L2",  "Last-digit (corr null)",  r2["p"]),
        _pline("L3",  "Draw sum distrib",         r3["p"]),
        _pline("L4",  "Odd/even split",           r4[list(r4)[0]]["p"]),
        _pline("L4",  "High/low split",           r4[list(r4)[1 if len(r4)>1 else 0]]["p"]),
        _pline("L5",  "Pair co-occurrence",       r5["p"]),
        _pline("L6",  "0s & 5s bias",             r6["p"]),
        _pline("L7",  "LD mean (corr null)",      r7["p_mean"]),
        _pline("L7",  "LD variance",              r7["p_var"]),
        _pline("L8",  "Sim-adj chi-square",       r8["p_empirical"]),
        _pline("L9",  f"Autocorr (max_lag={r9['max_lag']})", r9["lb_p"]),
        _pline("L10", "Variance — sum",           r10["sum"]["p"]),
        _pline("L10", "Variance — odd count",     r10["odd"]["p"]),
        _pline("L10", "Variance — range",         r10["range"]["p"]),
        _pline("L11", "Cross-ball correlation",   r11["p_chi2"]),
    ]
    n_fail = sum(1 for ln in lines[1:] if "✗" in ln)
    lines += ["", f"  Failures: {n_fail} / {len(lines)-1}", f"  Draws analysed: {n_draws:,}", f"  Range: {range_label if range_label else 'all'}"]
    ax.text(0.03, 0.97, "\n".join(lines), transform=ax.transAxes, fontsize=8.5, verticalalignment="top", fontfamily="monospace", bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.75))
    ax.set_title("Test Summary", fontsize=9)

    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    print(f"\n  Dashboard saved → '{out_path}'")

def save_report(logs: list, out_path: str = "lotto_forensics_report.txt") -> None:
    with open(out_path, "w", encoding="utf-8") as f:
        f.write("\n".join(["=" * 65, "  KOREAN LOTTO 6/45 — STATISTICAL FAIRNESS AUDIT REPORT", "=" * 65, ""] + logs))
    print(f"  Text report saved → '{out_path}'")

# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────
def parse_args():
    parser = argparse.ArgumentParser(
        description="Korean Lotto 6/45 Statistical Fairness Audit"
    )
    parser.add_argument("csv", nargs="?", default=None, help="Path to CSV file (omit for synthetic demo)")
    parser.add_argument("--from", dest="draw_from", type=int, default=None, help="First draw_no to include")
    parser.add_argument("--to",   dest="draw_to",   type=int, default=None, help="Last draw_no to include")
    parser.add_argument("--lag",  dest="lag",       type=int, default=5,    help="Max lag period for L9 Autocorrelation test (default: 5)")
    return parser.parse_args()

def main() -> None:
    args = parse_args()

    if args.csv:
        df, ball_cols, bonus_col = load_lotto_data(args.csv)
        df = filter_by_draw_range(df, args.draw_from, args.draw_to)
        draw_col = next((c for c in ["draw_no", "회차", "round", "draw"] if c in df.columns), None)
        if draw_col:
            lo, hi = int(df[draw_col].min()), int(df[draw_col].max())
            range_label, range_tag = f"draws {lo}–{hi}", f"{lo}_{hi}"
        else:
            range_label, range_tag = "", "all"
        tag = f"{args.csv.replace('.csv', '').replace('.CSV', '')}_{range_tag}"
    else:
        print("\n  [!] No data file provided — using synthetic fair draws (demo).")
        df, ball_cols, bonus_col = generate_synthetic_data(n_draws=1100)
        range_label, tag = "synthetic demo", "synthetic_demo"

    balls = extract_balls(df, ball_cols)
    logs = []

    print(f"\n  Running forensic tests on {len(balls):,} draws...")
    results = {
        "L1":  test_L1_ball_frequency(balls, logs),
        "L2":  test_L2_last_digit(balls, logs),
        "L3":  test_L3_draw_sum(balls, logs),
        "L4":  test_L4_splits(balls, logs),
        "L5":  test_L5_pairwise(balls, logs),
        "L6":  test_L6_zeros_fives(balls, logs),
        "L7":  test_L7_ld_mean_var(balls, logs),
        "L8":  test_L8_sim_chi2(balls, logs, n_sim=500, batch=100),
        "L9":  test_L9_autocorrelation(balls, logs, max_lag=args.lag),
        "L10": test_L10_variance(balls, logs),
        "L11": test_L11_cross_correlation(balls, logs),
    }

    plot_dashboard(balls, results, out_path=f"lotto_forensics_dashboard_{tag}.png", range_label=range_label)
    save_report(logs, out_path=f"lotto_forensics_report_{tag}.txt")

if __name__ == "__main__":
    main()

Writing lotto_forensics_new.py


In [10]:
!python lotto_forensics_new.py lotto_data.csv --from 1 --to 1204 --lag 100

  Loaded 1,204 draws | balls: ['b1', 'b2', 'b3', 'b4', 'b5', 'b6'] | bonus: bonus
  Filtered to draws 1–1204  (1,204 draws retained)

  Running forensic tests on 1,204 draws...

  [L1] Ball-Frequency Chi-Square  χ²(44)
  N draws              : 1,204
  Expected per ball    : 160.53
  χ²(44) = 29.5839   p = 0.9529
  Result: PASS
  Top-3 outlier balls (|z|-ranked):
    Ball  9: count= 132  z=-2.419
    Ball 22: count= 140  z=-1.741
    Ball 34: count= 181  z=+1.735

  [L2] Last-Digit Test — Corrected Null for 1–45
  N ball appearances   : 7,224
  χ²(9) = 6.9702   p = 0.6402
  Result: PASS
  digit  observed  expected  ratio
    0       648     642.1  1.009
    1       798     802.7  0.994
    2       760     802.7  0.947
    3       820     802.7  1.022
    4       829     802.7  1.033
    5       790     802.7  0.984
    6       649     642.1  1.011
    7       676     642.1  1.053
    8       641     642.1  0.998
    9       613     642.1  0.955

  [L3] Draw-Sum Distribution Test
  N dra